In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("F1-Strategy-Engine")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_master_lap_dataset"

df = spark.table(MASTER_PATH)

# Minimal sanity filtering
df = df.filter(
    (col("LapTime").isNotNull()) &
    (col("LapNumber").isNotNull()) &
    (col("DriverNumber").isNotNull())
)

df.printSchema()
df.count()

In [0]:
display(df)

In [0]:
df.columns

In [0]:
# Select relevant columns for ML modeling
ml_columns = [
    # Driver style
    "Driver", "LapTime", "LapNumber", "Stint", "Compound", "TyreLife", "FreshTyre", "HasPit",
    "GapToAhead", "DeltaToLeader", "Sector1Time", "Sector2Time", "Sector3Time",
    # Car performance
    "TeamName", "Year", "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST", "AirTemp_C", "TrackTemp_C", "Humidity_pct", "WindSpeed_kmh",
    # Tyre behavior
    "PitDuration",
    # Race context
    "FinalPosition", "Position", "QualiPosition", "TrackStatus", "Circuit"
]
df_ml = df.select(*ml_columns)
display(df_ml.limit(5))

In [0]:
from pyspark.sql.functions import when
# Convert boolean columns to integers
bool_cols = ["FreshTyre", "HasPit"]
for c in bool_cols:
    df_ml = df_ml.withColumn(c, when(col(c) == True, 1).otherwise(0))
display(df_ml.limit(5))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import last, first, col

# Fill Stint: forward fill within each Driver's race
window_stint = (
    Window.partitionBy("Driver")
    .orderBy("LapNumber")
    .rowsBetween(Window.unboundedPreceding, 0)
)
df_ml = df_ml.withColumn(
    "Stint",
    when(col("LapNumber") == 1, 1).otherwise(
        last("Stint", ignorenulls=True).over(window_stint)
    ),
)

# Fill TyreLife: forward fill, then set to 1 for first lap of each stint, increment for subsequent laps
window_tyre = Window.partitionBy("Driver", "Stint").orderBy("LapNumber")
df_ml = df_ml.withColumn(
    "TyreLife", last("TyreLife", ignorenulls=True).over(window_tyre)
)
df_ml = df_ml.withColumn(
    "TyreLife", col("LapNumber") - first("LapNumber").over(window_tyre) + 1
)

# Fill Compound: forward fill within each Driver's stint
from pyspark.sql.functions import coalesce, lit

window_compound = (
    Window.partitionBy("Driver", "Stint")
    .orderBy("LapNumber")
    .rowsBetween(Window.unboundedPreceding, 0)
)
first_compound = first("Compound", ignorenulls=True).over(window_compound)
last_compound = last("Compound", ignorenulls=True).over(window_compound)

from pyspark.sql.functions import count, sum as spark_sum

# Identify windows where all Compound values are null/"UNKNOWN"/"nan"/"None"
window_compound = Window.partitionBy("Driver", "Stint")

df_ml = df_ml.withColumn(
    "Compound",
    coalesce(
        when(
            (col("LapNumber") == 1)
            | col("Compound").isNull()
            | (col("Compound") == "UNKNOWN")
            | (col("Compound") == "nan")
            | (col("Compound") == "None"),
            coalesce(first_compound, last_compound, lit("SOFT")),
        ).otherwise(col("Compound")),
        lit("SOFT"),
    ),
)

df_ml = df_ml.withColumn(
    "Compound", when(col("Compound") == "nan", lit("SOFT")).otherwise(col("Compound"))
)

display(df_ml)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col, when, lit

speed_cols = ["SpeedI1", "SpeedI2", "SpeedFL", "SpeedST"]

window = Window.partitionBy("Year", "Circuit", "Driver")

for c in speed_cols:
    df_ml = df_ml.withColumn(
        c,
        when(
            col(c).isNull(),
            avg(col(c)).over(window)
        ).otherwise(col(c))
    )

df_ml = df_ml.withColumn(
    "TrackStatus",
    when(col("TrackStatus").isNull() | (col("TrackStatus") == ""), lit("1")).otherwise(col("TrackStatus"))
)

display(df_ml)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ----------------------------
# 1. Get Lap 1 position per driver per race
# ----------------------------
w_driver_race = Window.partitionBy("Year", "Circuit", "Driver")

df_ml = (
    df_ml
    .withColumn(
        "position_lap1",
        F.when(F.col("LapNumber") == 1, F.col("Position"))
    )
    .withColumn(
        "position_lap1",
        F.first("position_lap1", ignorenulls=True).over(w_driver_race)
    )
)

# ----------------------------
# 2. Collect used quali positions per race
# ----------------------------
w_race = Window.partitionBy("Year", "Circuit")

df_ml = df_ml.withColumn(
    "used_quali_positions",
    F.collect_set("QualiPosition").over(w_race)
)

# ----------------------------
# 3. Compute missing quali candidates (1–20)
# ----------------------------
full_quali_set = F.array(*[F.lit(i) for i in range(1, 21)])

df_ml = df_ml.withColumn(
    "missing_quali_candidates",
    F.array_except(full_quali_set, F.col("used_quali_positions"))
)

# ----------------------------
# 4. Choose missing quali closest to Lap 1 position
# ----------------------------
df_ml = df_ml.withColumn(
    "best_quali_fill",
    F.expr("""
        CAST(
            aggregate(
                missing_quali_candidates,
                CAST(NULL AS DOUBLE),
                (best, x) ->
                    IF(
                        best IS NULL OR abs(x - position_lap1) < abs(best - position_lap1),
                        CAST(x AS DOUBLE),
                        best
                    )
            ) AS INT
        )
    """)
)

# ----------------------------
# 5. Fill QualiPosition and clean up
# ----------------------------
df_ml = (
    df_ml
    .withColumn(
        "QualiPosition",
        F.when(
            F.col("QualiPosition").isNull(),
            F.col("best_quali_fill")
        ).otherwise(F.col("QualiPosition"))
    )
    .drop(
        "used_quali_positions",
        "missing_quali_candidates",
        "best_quali_fill",
        "position_lap1"
    )
)

display(df_ml)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, lag, lead, avg

# Proper windows
window_lap = Window.partitionBy("Year", "Circuit", "Driver").orderBy("LapNumber")
window_group = Window.partitionBy("Year", "Circuit", "LapNumber")

# Neighbors
prev_gap = lag("GapToAhead", 1).over(window_lap)
next_gap = lead("GapToAhead", 1).over(window_lap)
mean_gap = avg(col("GapToAhead")).over(window_group)

# Correct interpolation
df_ml = df_ml.withColumn(
    "GapToAhead",
    when(
        col("GapToAhead").isNull(),
        when(prev_gap.isNotNull() & next_gap.isNotNull(), (prev_gap + next_gap) / 2)
        .when(prev_gap.isNotNull(), prev_gap)
        .when(next_gap.isNotNull(), next_gap)
        .otherwise(mean_gap)
    ).otherwise(col("GapToAhead"))
)

display(df_ml)


In [0]:
from pyspark.sql import functions as F

# 1. Compute NULL counts for all columns
null_counts_df = df_ml.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_ml.columns
])

# 2. Keep only columns with NULLs
null_counts_df = null_counts_df.select([
    c for c in null_counts_df.columns
    if null_counts_df.select(F.col(c)).first()[0] > 0
])

display(null_counts_df)

In [0]:
# Window defining a single race
race_window = Window.partitionBy("Year", "Circuit")

df_ml = df_ml.withColumn(
    "RacePhase",
    F.when(F.col("LapNumber") <= 3, "START")
     .when(
         F.col("LapNumber") >= F.max("LapNumber").over(race_window) - 3,
         "END"
     )
     .otherwise("MID")
)

display(df_ml)

In [0]:
df_ml.write.mode("overwrite").saveAsTable("workspace.default.f1_cleaned_lap_dataset")